In [1]:
import json
import random
from pathlib import Path
from collections import defaultdict

SEED = 42
random.seed(SEED)

# -------------------------
# Paths (CHANGE THESE)
# -------------------------
WIKISOURCE_PATH = "data/wikisource_pd_scripts.jsonl"          # could be .jsonl too
SYNTHETIC_DIR   = "data/synthetic"               # folder of 1000 *.json
ERRORS_JSONL    = "data/labeled_error_cases.jsonl"

OUT_DIR = Path("data/splits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Helpers
# -------------------------
TEXT_KEYS = ["text", "script", "content", "screenplay", "raw_text"]
LIST_KEYS = ["scripts", "data", "items", "examples", "records"]

def extract_text(obj):
    if isinstance(obj, str):
        t = obj.strip()
        return t if t else None
    if isinstance(obj, dict):
        for k in TEXT_KEYS:
            v = obj.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
        # fallback: longest string field
        strs = [v.strip() for v in obj.values() if isinstance(v, str) and v.strip()]
        if strs:
            return max(strs, key=len)
    return None

def load_json_or_jsonl(path: str):
    """Loads a file that might be JSON (single obj/list) or JSONL (one obj per line)."""
    p = Path(path)
    text = p.read_text(encoding="utf-8", errors="replace")

    # Try normal JSON first
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            # if dict contains a list under some key
            for k in LIST_KEYS:
                if k in obj and isinstance(obj[k], list):
                    return obj[k]
            return [obj]
    except json.JSONDecodeError:
        pass

    # Fallback JSONL
    rows = []
    with open(p, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

#def stratified_split(rows, label_key: str, train=0.9, val=0.05, test=0.05, seed=42):
def stratified_split(rows, label_key: str, train=0.75, val=0.15, test=0.1, seed=42):
    assert abs(train + val + test - 1.0) < 1e-9
    rng = random.Random(seed)

    buckets = defaultdict(list)
    for r in rows:
        buckets[r[label_key]].append(r)

    train_rows, val_rows, test_rows = [], [], []
    for k, group in buckets.items():
        rng.shuffle(group)
        n = len(group)
        n_train = int(n * train)
        n_val = int(n * val)
        train_rows.extend(group[:n_train])
        val_rows.extend(group[n_train:n_train + n_val])
        test_rows.extend(group[n_train + n_val:])

    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    rng.shuffle(test_rows)
    return train_rows, val_rows, test_rows

# -------------------------
# Load Wikisource scripts
# -------------------------
wik_raw = load_json_or_jsonl(WIKISOURCE_PATH)
wikisource = []
for i, obj in enumerate(wik_raw):
    text = extract_text(obj)
    if text and len(text) > 500:
        wikisource.append({
            "id": f"wikisource_{i}",
            "dataset": "wikisource_clean",
            "text": text
        })

print("Wikisource clean:", len(wikisource))

# -------------------------
# Load 1000 synthetic clean scripts (each file is one script JSON)
# -------------------------
synthetic = []
for p in sorted(Path(SYNTHETIC_DIR).glob("*.json")):
    try:
        obj = json.loads(p.read_text(encoding="utf-8", errors="replace"))
    except json.JSONDecodeError:
        continue
    text = extract_text(obj)
    if text and len(text) > 500:
        synthetic.append({
            "id": f"synthetic_{p.stem}",
            "dataset": "synthetic_clean",
            "text": text
        })

print("Synthetic clean:", len(synthetic))

# -------------------------
# Load labeled error cases (already JSONL)
# We keep both clean+corrupted because you may train different objectives later.
# -------------------------
err_rows = load_json_or_jsonl(ERRORS_JSONL)
errors = []
for i, obj in enumerate(err_rows):
    # expected fields from earlier generator; tolerate missing fields
    clean_ex = obj.get("clean_excerpt")
    bad_ex = obj.get("corrupted_excerpt")
    labels = obj.get("labels", [])
    expl = obj.get("explanation", "")
    fix = obj.get("suggested_fix", "")

    if isinstance(clean_ex, str) and isinstance(bad_ex, str) and clean_ex.strip() and bad_ex.strip():
        errors.append({
            "id": obj.get("id", f"error_{i}"),
            "dataset": "labeled_errors",
            "labels": labels,
            "clean_excerpt": clean_ex.strip(),
            "corrupted_excerpt": bad_ex.strip(),
            "explanation": expl,
            "suggested_fix": fix
        })

print("Labeled errors:", len(errors))

# -------------------------
# Combine and split (stratified by dataset)
# -------------------------
all_rows = wikisource + synthetic + errors
print("TOTAL rows:", len(all_rows))

#train_rows, val_rows, test_rows = stratified_split(all_rows, label_key="dataset",
#                                                   train=0.90, val=0.05, test=0.05,
#                                                   seed=SEED)

train_rows, val_rows, test_rows = stratified_split(all_rows, label_key="dataset",
                                                   train=0.75, val=0.15, test=0.1,
                                                   seed=SEED)

print(f"TRAIN={len(train_rows)}  VAL={len(val_rows)}  TEST={len(test_rows)}")

# -------------------------
# Write splits
# -------------------------
write_jsonl(OUT_DIR / "train.jsonl", train_rows)
write_jsonl(OUT_DIR / "val.jsonl", val_rows)
write_jsonl(OUT_DIR / "test.jsonl", test_rows)

print("Wrote:", OUT_DIR / "train.jsonl")
print("Wrote:", OUT_DIR / "val.jsonl")
print("Wrote:", OUT_DIR / "test.jsonl")

Wikisource clean: 49
Synthetic clean: 1000
Labeled errors: 550
TOTAL rows: 1599
TRAIN=1198  VAL=239  TEST=162
Wrote: data/splits/train.jsonl
Wrote: data/splits/val.jsonl
Wrote: data/splits/test.jsonl
